Kept so the pilot data can be rebuilt. This notebook is NOT run in part four, the data stays where it is on Drive and in its private repo, both named by the .env file.

# Stage A, part 1 - the pilot data

Stage A trains two students on a pilot of about 20,000 Major-TOM pairs (26,000 cells are drawn, the pairing rules keep about 79 percent of them by the stage 0 counts) and 20,000 FLAIR-HUB aerial patches. This notebook fetches that data and puts it on Drive. Nothing is trained and nothing is scored here. The fixed values come from the code (pilotdata, records, aerial), the notebook prints them and never retypes them.

## What it does

- the index of all Major-TOM cells (the index files of stage 0 on Drive are reused) and the pairs within 7 days
- the pilot sample, 26,000 cells drawn with seed 0 and spread over the years, sorted by shard and cut into 13 chunks of 2,000
- the fetch, chunk by chunk with 28 threads. The rules are applied once, when a pair is made (cloud codes 1, 2, 3 over 10 percent, no-data in either sensor, the offset by processing baseline). Each chunk becomes one archive, built on the runtime disk and copied to Drive, with a JSON and an infos table next to it (chunk_k_infos.csv, one row per pair with its grid cell, dates, baseline, offset, cloud fraction, product ids and its place in the tar). The local pair files of a chunk are deleted once its archive is complete on Drive, so the runtime disk holds one chunk at a time. An archive is never unpacked, every later cell reads a pair from it in place
- 2,000 pairs held out for the distillation loss, drawn with seed 0 on the grid cells of the infos
- the aerial sets of FLAIR-HUB, planned from the zip directories before any patch is read, then built by range requests with 28 threads, halved to 0.4 m, one archive per set on Drive
- the channel statistics of the student, computed once from 2,000 training pairs (read from the archives in place) and the aerial training images
- a summary of the pilot from the infos tables
- the upload of everything the next notebooks read to the private Hugging Face repo named by DATA_HF_REPO in the .env file, with the write token HF_WRITE_TOKEN from the same file, as part two did with its converted datasets, 4 files at once, each file its own upload. The code writes a card with the licences and the attribution of the sources, creates the repo private and refuses a public one. A file already there with the same size is skipped, so the cell can be run again after a break. Without the token the cell says so and goes on, Drive holds everything
- a flush of Drive at the end. The run is over when the last cell prints, Drive then holds everything. Drive is unmounted by it, run the mount cell again to use it

## Runtime

A standard runtime, no GPU. It has about 12.7 GB RAM and about 100 GB disk. No cell loads all pairs into memory, the pairs stay in their archives. The notebook is opened from VS Code with the Colab extension.

## Run time and download

Measured on 2026-09-30 with 16 threads and the read token, on a standard runtime. Every chunk of 2,000 cells took 470 to 576 s, about 4 cells per second and about 8 requests per second (about 2,400 per 5 minutes, no wait from the Hub), 23 GB of rows and 200 to 400 MB of footers per chunk, so about 12 MB per cell and about 300 GB for the 26,000 cells. 11 of 13 chunks (17,106 pairs, 41 GB of archives) were on Drive after about 95 minutes when the session dropped. Stage 0 had read 16 MB per cell (11.5 MB of rows, the rest footers), about 7 s per cell per thread, 1.1 cells per second with 28 threads, the network was the limit. The fetch now runs 28 threads. The Hub allows 5,000 requests per 5 minutes with a token (3,000 anonymous), 28 threads aim at about 14 requests per second and about 7 cells per second, so the 2 missing chunks take about 10 minutes and all 13 about 1 hour if the speed per thread holds. That is an estimate, the fetch prints the requests per second and the waits (majortom.STATS wait_seconds and retries) per chunk, so the real limit shows. About 79 percent of the drawn cells become pairs by the stage 0 counts (393 of 500, the cloud rule and the Sentinel-2 no-data rule take most of the rest), the real run kept 78 percent of the first 22,000 cells, so about 20,000 pairs are expected. The archives on Drive are about 2.4 MB per pair at 384 px, about 49 GB and up to 62 GB. The aerial sets are about 24 GB in 32,000 range requests, with 28 threads about 11 to 13 requests per second, under the limit with the token, so about about 1.5 hours, measured 4.5 hours with 8 threads, about 8 GB on disk after halving, and the same on Drive. The runtime disk holds one chunk of pairs (about 5 GB) and its tar while it is copied, and the aerial sets, about 20 GB in all. A first run writes about 57 GB to Drive, up to 70 GB. On a re-run nothing is fetched or unpacked for a chunk whose archive is on Drive, it costs a few seconds (its infos table is read). A chunk with an archive but no infos table (the 11 archives of the first run) gets one built from the tar once, the info entry of every member is read in place, the time is printed. The setup cell warns when the disk or Drive has less room. The upload sends the same 57 to 70 GB to Hugging Face, read from Drive, 4 files at once, its rate is not measured, the cell prints the MB per second per file and in all and the time. Every cell prints its time.

## Results

Google Drive, the folder DATA_DRIVE_DIR of the .env file under MyDrive, pilot/ inside it. In majortom/ the chunk archives with their JSON files and infos tables, the sample, the skip and timing tables, the two held-out lists and the channel statistics. In aerial/ one tar per set, the manifest, the plan and the two cached listings. The summary tables are in the folder itself, the card README.md of the repo too. The same files go to the private Hugging Face repo when the write token is there, notebooks 02 and 03 read from the repo first and from Drive when the repo is missing. The working copies are under /content, in a folder of the same name, and are lost with the runtime.

## How to run

Run the cells from top to bottom. After a crash or on a new runtime run them again from the top. A chunk whose archive is on Drive is read in place and not fetched again, its infos table is built once when it is missing, an aerial set whose archive is on Drive is unpacked, the statistics are computed once. To fetch a chunk again delete its archive and its JSON file on Drive. The sample and the plan of a run are compared with the ones on Drive, a difference stops the notebook before anything is fetched. A cell that could not be read (read_error) stays skipped inside a complete archive. The fetch cell names such chunks, delete their archive and JSON on Drive to read them again. The last cell waits until Drive holds everything and unmounts it. Run the mount cell again to do anything else on Drive afterwards.

## Token

The Hugging Face token is read from the .env file and is passed to the Major-TOM and FLAIR-HUB reads for the higher rate limit. The write token HF_WRITE_TOKEN of the same file, when it is there, uploads the pilot to the private repo. The tokens are never printed. The two data settings, DATA_HF_REPO (the private repo of the pilot data, owner/name) and DATA_DRIVE_DIR (the folder under MyDrive that holds pilot/ and stage0/), come from the same file, so this notebook names no folder and no repo.


## 1. Mount Google Drive

Drive holds the archives and the tables. Everything that must survive the runtime goes there.

In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


## 2. Unpack the code

The package source travels inside this notebook. This cell unpacks it on the runtime, so no clone and no token are needed for the code.

In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

## 3. Setup

Reads the tokens from the .env file, installs rasterio if it is missing, prints RAM, disk and the fixed settings of the pilot (cells drawn, cloud codes, offset rule, aerial set sizes, threads). Nothing here is retyped, every number comes from the modules.

In [ ]:
# Setup. Libraries, token, settings, folders. This notebook needs no GPU.
import contextlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

import pandas as pd

from sartransfer import env
from sartransfer.distil import aerial, majortom, pairing, pilotdata, stage0
from sartransfer.distil.records import CLOUD_CODES, SKIP_REASONS
from sartransfer.domains import hub

start = time.perf_counter()
CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
assert MYDRIVE.is_dir(), "Google Drive is not mounted, run the mount cell first"

for module in ("rasterio",):                   # installed only when the runtime lacks it
    if importlib.util.find_spec(module) is None:
        print("installing", module)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", module], check=True)
        importlib.invalidate_caches()

env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the Major-TOM and FLAIR-HUB reads")
assert HF_TOKEN, "no Hugging Face token was given"
# Where the data is, from the same .env file. No folder and no repo is written into this notebook.
DATA_DIR = env.require("DATA_DRIVE_DIR", "the folder under MyDrive that holds pilot/ and stage0/")
assert DATA_DIR, "no data folder was given"
DATA_REPO = hub.data_repo_id()         # the private Hugging Face repo of the pilot data, owner/name

# Settings of this notebook, how the work is done. None of them is a rule.
# WORKERS. The read token goes with every Major-TOM read. The Hub limit with a token was read as 5,000
# requests per 5 minutes (3,000 anonymous). The run of 2026-09-30 at 16 threads used about 2,400 per
# 5 minutes (8 per second, 4 cells per second). 28 threads aim at about 14 per second, still under the
# limit. The fetch prints the requests per second and the waits per chunk, so the real limit shows.
WORKERS = 28                     # threads of the Major-TOM fetch (pilotdata.WORKERS says why)
AERIAL_WORKERS = 28   # measured 2026-10-01 with 8 threads, about 4 s per range request and 2 requests per second, no waits, so the wall is latency and not the request limit
DRIVE_NEED_GB = 70               # up to 62 GB of pair archives and 8 GB of aerial archives
DISK_NEED_GB = 20                # the runtime disk, one chunk of pairs (5 GB) and its tar while it is copied, 8 GB of aerial sets

# The fixed values come from the code. They are printed, never retyped.
N_PILOT, N_HELD_OUT, N_STATS, SEED = (pilotdata.N_PILOT, pilotdata.N_HELD_OUT, pilotdata.N_STATS,
                                      pilotdata.SEED)
print(f"pilot {N_PILOT:,} cells drawn for about {pilotdata.N_PAIRS:,} pairs, {N_HELD_OUT:,} held out, "
      f"statistics over {N_STATS:,} pairs, seed {SEED}, chunks of {pilotdata.CHUNK:,} cells")
print(f"cloud codes {list(CLOUD_CODES)} over {stage0.MAX_CLOUD:.0%} of the crop, offset by "
      f"processing baseline, pairs within {stage0.MAX_GAP_DAYS:g} days")
print(f"aerial sets {aerial.SET_SIZES}, seeds {aerial.SET_SEEDS}, {aerial.PATCH_PX} px halved to "
      f"{aerial.OUT_PX} px at {aerial.PIXEL_M} m")

PILOT = MYDRIVE / DATA_DIR / "pilot"                     # on Drive, what the next notebooks read
DRIVE_MT, DRIVE_AERIAL = PILOT / "majortom", PILOT / "aerial"
STAGE0 = MYDRIVE / DATA_DIR / "stage0"                   # the index files of stage 0 are reused
WORK = CONTENT / DATA_DIR / "pilot"                      # runtime disk, lost with the runtime
WORK_MT, WORK_AERIAL = WORK / "majortom", WORK / "aerial"
SETS_DIR = WORK_AERIAL / "sets"
for folder in (DRIVE_MT, DRIVE_AERIAL, WORK_MT, SETS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

TIMES = {}                       # seconds of every step of this run, by name


def num(value, digits=2):
    """A number as text, "not known" for None and nan."""
    if value is None or value != value:
        return "not known"
    return f"{value:,.{digits}f}"


def show(table):
    """Print a whole table."""
    print(table.to_string(index=False))


def hide(text):
    """A text without the token."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def took(since, name):
    """The time of a cell as text, recorded under `name` for the summary."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def paid_since(before):
    """What majortom counted since `before` (a copy of majortom.STATS), as text with the rate."""
    paid = {k: majortom.STATS[k] - before[k] for k in majortom.STATS}
    seconds = time.perf_counter() - before["_clock"]
    return (f"{paid['requests']:,.0f} requests, {paid['requests'] / max(seconds, 1e-9):.1f} per second, "
            f"waited {paid['wait_seconds']:,.0f} s in {paid['retries']:,.0f} retries, "
            f"{paid['bytes'] / stage0.MB:,.0f} MB in {seconds:,.0f} s")


runtime = stage0.runtime_report([WORK, PILOT])
print(f"GPU     {runtime['gpu'] or 'none, none is needed here'}"
      + (", this notebook needs no GPU, a standard runtime is enough" if runtime["gpu"] else ""))
print(f"RAM     {num(runtime['ram_gb'], 1)} GB, free {num(runtime['ram_free_gb'], 1)} GB, "
      f"{runtime['cpus']} CPUs")
for folder, disk in runtime["disk"].items():
    print(f"disk    {folder}, free {num(disk['free_gb'], 1)} of {num(disk['total_gb'], 1)} GB")
drive_free = runtime["disk"][str(PILOT)]["free_gb"]
if drive_free is not None and drive_free < DRIVE_NEED_GB:
    print(f"WARNING Drive has {drive_free:.1f} GB free. A first run writes about {DRIVE_NEED_GB} GB "
          "there (pair archives, aerial archives). Make room on Drive before the cells below are run")
disk_free = runtime["disk"][str(WORK)]["free_gb"]
if disk_free is not None and disk_free < DISK_NEED_GB:
    print(f"WARNING the runtime disk has {disk_free:.1f} GB free. A run keeps up to {DISK_NEED_GB} GB "
          "there (one chunk of pairs and its tar while it is copied, the aerial sets). A full disk turns "
          "every later cell into a read_error skip. Use a runtime with more disk")
print(f"results go to   {PILOT}")
print(f"work folder     {WORK}")
stage0.save_report(PILOT, runtime=runtime)
print(took(start, "setup"))


## 4. Index of the Major-TOM cells

Loads the two lists of all cells (made in stage 0, reused from Drive), joins them by grid cell and applies the 7 day rule. Result about 150,000 usable pairs.

In [ ]:
# Index. The two index files of stage 0 on Drive are reused, else built here and kept. Then the pairs.
start = time.perf_counter()
index_files = {}
for name in ("s2", "s1"):
    of_stage0, own = STAGE0 / f"index_{name}.parquet", PILOT / f"index_{name}.parquet"
    index_files[name] = of_stage0 if of_stage0.is_file() else own
    where = ("found on Drive, from stage 0" if of_stage0.is_file() else
             "found on Drive, own copy" if own.is_file() else "not on Drive, built now")
    print(f"index {name}  {where}  {index_files[name]}")
clock = time.perf_counter()
s2_index = majortom.build_s2_index(cache=index_files["s2"], token=HF_TOKEN)
s1_index = majortom.load_s1_index(cache=index_files["s1"], token=HF_TOKEN)
print(f"index read in {time.perf_counter() - clock:.1f} s, Sentinel-2 {len(s2_index):,} rows in "
      f"{s2_index['parquet_url'].nunique():,} shards, Sentinel-1 {len(s1_index):,} rows")

pairs, pair_counts = pairing.pair_index(s2_index, s1_index, max_gap_days=stage0.MAX_GAP_DAYS)
print(f"Sentinel-2 grid cells          {pair_counts['s2_cells']:>10,}")
print(f"grid cells with Sentinel-1     {pair_counts['cells_with_s1']:>10,}")
print("grid cells left out by the index, per skip reason")
for reason in ("no_s1_for_cell", "gap_gt_7d"):
    print(f"  {reason:<28} {pair_counts[reason]:>10,}")
print(f"pairs within {stage0.MAX_GAP_DAYS:g} days           {pair_counts['pairs']:>10,}")
assert len(pairs), "the index holds no pair"
stage0.save_report(PILOT, pair_counts=pair_counts)
del s2_index, s1_index
print(took(start, "index"))


## 5. The pilot sample

Draws 26,000 cells with seed 0, spread over the years, sorted so that neighbours share a file on Hugging Face, in 13 chunks of 2,000. A saved sample that differs from a fresh draw stops the notebook.

In [ ]:
# Pilot sample. The draw, in the order of the shards, cut into chunks. The
# sample on Drive is compared with the draw of this run, a difference stops the notebook.
start = time.perf_counter()
sample = pilotdata.pilot_sample(pairs, n=N_PILOT, seed=SEED)
assert len(sample), "the index holds no pair"
SAMPLE = DRIVE_MT / "pilot_sample.csv"
if SAMPLE.is_file():
    saved = pd.read_csv(SAMPLE, dtype={"grid_cell": str})
    same = (len(saved) == len(sample) and saved["grid_cell"].tolist() == sample["grid_cell"].tolist()
            and saved["chunk"].tolist() == sample["chunk"].tolist())
    assert same, (f"{SAMPLE} holds another sample than this run draws, so the index changed. "
                  "Nothing is fetched. The archives on Drive belong to the saved sample. Decide "
                  "which sample counts, then delete the other one and run the cell again")
    print(f"the sample on Drive is the same draw, {len(sample):,} cells")
else:
    stage0.save_report(DRIVE_MT, pilot_sample=sample)
    print(f"sample of {len(sample):,} cells written to {SAMPLE.name}")
print(f"seed {SEED}, {len(sample):,} of {len(pairs):,} pairs, cells per year")
print(sample.groupby("s2_year").size().to_string(header=False))
per_chunk = sample.groupby("chunk").agg(cells=("grid_cell", "size"), s2_shards=("s2_url", "nunique"),
                                        s1_shards=("s1_url", "nunique")).reset_index()
print(f"{len(per_chunk)} chunks of up to {pilotdata.CHUNK:,} cells, cells and shards per chunk")
show(per_chunk)
print(f"shards of the sample, Sentinel-2 {sample['s2_url'].nunique():,}, "
      f"Sentinel-1 {sample['s1_url'].nunique():,}")
print(took(start, "sample"))


## 6. Fetch the pairs

The long cell. For each cell the threads read the Sentinel-2 crop and the Sentinel-1 cell from Hugging Face, cut the radar to the optical footprint, apply the rules and save one file per pair. Each finished chunk becomes one archive on Drive with a JSON and a small table of pair facts. On a re-run finished chunks are found on Drive and nothing is unpacked.

In [ ]:
# Fetch. Chunk by chunk, the rules applied once when a pair is made, each chunk one archive on Drive
# with its JSON and its infos table. A chunk whose archive is on Drive is read in place, never unpacked
# and never fetched again (its infos table is built once when missing). The local pair files of a chunk
# are deleted once its archive is complete, so the runtime disk holds one chunk at a time. Resumable.
start = time.perf_counter()
print(f"{WORKERS} threads, cloud codes {list(CLOUD_CODES)}, offset by baseline, archives in {DRIVE_MT}")
paths, skips, timing = pilotdata.fetch_pilot(sample, WORK_MT, DRIVE_MT, workers=WORKERS,
                                             token=HF_TOKEN, cloud_codes=CLOUD_CODES,
                                             offset="baseline")
totals = timing.attrs
print(f"\n{len(paths):,} of {len(sample):,} cells became a pair (the plan asks for about "
      f"{pilotdata.N_PAIRS:,}), {totals['fetched']} chunks fetched now, {totals['from_archive']} "
      f"read from Drive in place, {totals['infos_built']} infos tables built, {totals['wall_s']:,.0f} s")
by_reason = skips["reason"].value_counts()
print("skipped cells per reason")
for reason in SKIP_REASONS:
    print(f"  {reason:<20} {int(by_reason.get(reason, 0)):>6}")
errors = skips[skips["reason"] == "read_error"]
if len(errors):
    print("the first read errors")
    show(errors.head(10).assign(detail=lambda t: t["detail"].map(hide)))
    chunks = ", ".join(str(k) for k in sorted(errors["chunk"].unique()))
    print(f"WARNING {len(errors):,} cells could not be read (chunks {chunks}). Their archives on Drive "
          "are complete, so these cells are not read again on their own. To read them again delete "
          f"chunk_<k>.tar and chunk_<k>.json of these chunks in {DRIVE_MT} and run this cell again")
fetched = timing[timing["source"] == "fetched"]
if len(fetched):
    cells, seconds = int(fetched["cells"].sum()), float(fetched["wall_s"].sum())
    requests = fetched["requests"].sum()
    print(f"\nthe chunks fetched in this run, {cells:,} cells in {seconds:,.0f} s, "
          f"{cells / max(seconds, 1e-9):.2f} cells per second, rows {fetched['mb_rows'].sum():,.0f} MB, "
          f"footers {fetched['mb_footers'].sum():,.0f} MB, {requests:,.0f} requests, "
          f"{requests / max(seconds, 1e-9):.1f} per second, waited {fetched['wait_s'].sum():,.0f} s in "
          f"{fetched['retries'].sum():,.0f} retries (the Hub allows 5,000 per 5 minutes with a token)")
stage0.save_report(DRIVE_MT, fetch_skips=skips, fetch_timing=timing)
assert paths, "no cell became a pair, see the skip table above"
# the infos tables of the chunks, one row per pair, what the held-out, statistics and summary cells read
infos = pilotdata.load_infos(DRIVE_MT, sample["chunk"].unique())
assert pilotdata.locations_of(infos, DRIVE_MT) == paths, "the infos tables do not list the fetched pairs"
print(f"infos of {len(infos):,} pairs in {infos['chunk'].nunique()} tables, "
      f"{infos['bytes'].sum() / stage0.GB:.1f} GB of pairs in the archives")
print(took(start, "fetch"))


## 7. Held-out pairs

2,000 pairs are set aside by seed for the distillation loss. Two lists of cell names on Drive.

In [ ]:
# Held-out split. The pairs for the distillation loss, drawn with the seed on the sorted grid cells of
# the infos. The two lists of cells go to Drive next to the archives.
start = time.perf_counter()
train_paths, held_paths = pilotdata.split_held_out(paths, n=N_HELD_OUT, seed=SEED,
                                                   archive_dir=DRIVE_MT)
print(f"{len(train_paths):,} training pairs, {len(held_paths):,} held out, seed {SEED}")
print(f"lists on Drive  {DRIVE_MT / pilotdata.TRAIN_FILE}  {DRIVE_MT / pilotdata.HELD_OUT_FILE}")
print(took(start, "held_out"))


## 8. Aerial plan

Lists the FLAIR-HUB zips, reads the table of contents of each zip by byte ranges, and draws the five aerial sets by seed. The plan is saved before any patch is read. A saved plan that differs stops the notebook.

In [ ]:
# Aerial plan. The zips of FLAIR-HUB, the patch index from their central directories (both cached on
# Drive), the five sets planned before any patch is read. A plan on Drive is compared with this draw.
start = time.perf_counter()
listing = aerial.list_zips(token=HF_TOKEN, cache=DRIVE_AERIAL / "zips.csv")
rgbi = listing[listing["modality"] == "AERIAL_RGBI"]
print(f"{len(listing)} aerial zips, {len(rgbi)} with images, {rgbi['bytes'].sum() / stage0.GB:,.1f} GB, "
      f"revision {aerial.HF_REVISION[:12]}")
patches = aerial.list_patches(listing, cache=DRIVE_AERIAL / "patches.csv", token=HF_TOKEN,
                              workers=AERIAL_WORKERS)
print(f"{len(patches):,} patches in the index, {int((patches['label_zip'] != '').sum()):,} with a label")
plan = aerial.plan_sets(patches)
PLAN_CSV = DRIVE_AERIAL / "plan.csv"
if PLAN_CSV.is_file():
    saved_plan = aerial.read_plan(PLAN_CSV)
    assert saved_plan.equals(plan), (
        f"{PLAN_CSV} holds another plan than this run draws, so the patch index changed. Nothing "
        "is read. The archives on Drive belong to the saved plan. Decide which plan counts, then "
        "delete the other one and run the cell again")
    print("the plan on Drive is the same draw")
else:
    aerial.write_plan(plan, PLAN_CSV)
    print(f"plan written to {PLAN_CSV.name}")
counts = plan.groupby("set", sort=False).agg(patches=("patch_id", "size"),
                                              departments=("department", "nunique"),
                                              zips=("rgbi_zip", "nunique")).reset_index()
show(counts)
print(f"{plan['patch_id'].nunique():,} distinct patches in {len(plan):,} rows (the probe draw may "
      "overlap the distillation draw, its labels are never used there)")
print(took(start, "aerial_plan"))


## 9. Aerial build

Reads only the planned patches by byte ranges, halves them to 256 px at 0.4 m, and writes one archive per set to Drive. A finished set is kept, an unfinished one is redone. The first run measured about 4 s per range request, so 28 threads read in parallel to hide that latency.


In [ ]:
# Aerial build. The planned patches by range requests, halved, one file per set and department on the
# runtime disk, then one archive per set on Drive. A set whose archive is on Drive is unpacked, not read again.
start = time.perf_counter()


def unpack_set(archive, folder):
    """The files of one set archive into `folder`, through a side folder. False when it cannot be read."""
    side = folder.parent / f"unpack_{archive.stem}"
    shutil.rmtree(side, ignore_errors=True)
    try:
        with tarfile.open(archive) as tar:
            try:
                tar.extractall(side, filter="data")
            except TypeError:                     # an older Python without the filter argument
                tar.extractall(side)
    except (tarfile.TarError, EOFError, OSError) as err:
        print(f"{archive.name} on Drive cannot be read, {type(err).__name__}, {hide(err)}. The set "
              "is built again and the archive is written again")
        shutil.rmtree(side, ignore_errors=True)
        return False
    for path in sorted(side.glob("*.npz")):
        os.replace(path, folder / path.name)
    shutil.rmtree(side, ignore_errors=True)
    return True


def pack_set(files, archive):
    """The files of one set into `archive` on Drive, through a local tar and a temporary name."""
    local, part = WORK_AERIAL / archive.name, archive.with_name(archive.name + ".part")
    try:
        with tarfile.open(local, "w") as tar:     # no compression, the files are not compressed either
            for file in files:
                tar.add(SETS_DIR / file, arcname=file)
        shutil.copyfile(local, part)
        os.replace(part, archive)
    finally:
        for path in (local, part):
            with contextlib.suppress(OSError):
                path.unlink()
    return archive.stat().st_size


MANIFEST = DRIVE_AERIAL / "manifest.json"
broken = set()                               # sets whose archive on Drive could not be read
if MANIFEST.is_file():                       # a new runtime, the sets come back from Drive
    saved = json.loads(MANIFEST.read_text(encoding="utf-8"))
    for name, entry in saved["sets"].items():
        archive = DRIVE_AERIAL / f"{name}.tar"
        if archive.is_file() and not all((SETS_DIR / f).is_file() for f in entry["files"]):
            clock = time.perf_counter()
            if unpack_set(archive, SETS_DIR):
                print(f"{name:<15} {len(entry['files'])} files unpacked from {archive.name} in "
                      f"{time.perf_counter() - clock:.1f} s")
            else:
                broken.add(name)
before = {**majortom.STATS, "_clock": time.perf_counter()}
manifest = aerial.build_sets(plan, WORK_AERIAL / "zips", SETS_DIR, token=HF_TOKEN, listing=listing,
                             how="ranges", workers=AERIAL_WORKERS)
print(f"range reads of this build with {AERIAL_WORKERS} threads and the token, {paid_since(before)}")
MANIFEST.write_text(json.dumps(manifest, indent=1), encoding="utf-8")
print()
for name in aerial.SETS:
    entry = manifest["sets"][name]
    archive = DRIVE_AERIAL / f"{name}.tar"
    mb = sum((SETS_DIR / f).stat().st_size for f in entry["files"]) / stage0.MB
    if archive.is_file() and name not in broken:
        print(f"{name:<15} {entry['count']:>6,} patches, {len(entry['files'])} files, {mb:,.0f} MB, "
              f"the archive is on Drive already")
        continue
    clock = time.perf_counter()
    size = pack_set(entry["files"], archive)
    print(f"{name:<15} {entry['count']:>6,} patches, {len(entry['files'])} files, {mb:,.0f} MB, copied to "
          f"Drive as {archive.name} ({size / stage0.MB:,.0f} MB) in {time.perf_counter() - clock:.1f} s")
print(f"manifest on Drive {MANIFEST}")
print(took(start, "aerial_build"))


## 10. Channel statistics

Mean and spread per channel over 2,000 training pairs and the aerial training images, once, as one JSON on Drive. Every student uses the same numbers.

In [ ]:
# Channel statistics. Mean and SD per channel from training pairs (read from the archives in place)
# and the aerial training images, computed once and kept on Drive. Every student is standardised with them.
start = time.perf_counter()
STATS = DRIVE_MT / "channel_stats.json"
if STATS.is_file():
    stats = json.loads(STATS.read_text(encoding="utf-8"))
    print(f"found on Drive, not computed again, {STATS.name}, {stats['n']:,} pairs and "
          f"{stats['aerial']['images']:,} aerial images")
else:
    distil_train = aerial.load_set(SETS_DIR, "distil_train")
    print(f"aerial training images {distil_train['images'].shape}, "
          f"{distil_train['images'].nbytes / stage0.GB:.2f} GB in RAM")
    stats = pilotdata.channel_stats(train_paths, n=N_STATS, seed=SEED, aerial=distil_train["images"],
                                    out_path=STATS, workers=AERIAL_WORKERS)
    del distil_train
    print(f"computed over {stats['n']:,} pairs (seed {stats['seed']}) and {stats['aerial']['images']:,} "
          f"aerial images, written to {STATS.name}")
print(f"corner rule applied to every pair  {stats['corner_rule']}")
for key in ("s1", "s2", "aerial"):
    part = stats[key]
    table = pd.DataFrame({"channel": part["channels"], "mean": part["mean"], "sd": part["sd"]})
    print(f"\n{key}, {part['unit']}")
    print(table.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
print(took(start, "stats"))


## 11. Summary

Counts per year, per skip reason and per processing version, the cloud share, sizes and the time of every step, from the pair tables. Saved on Drive.

In [ ]:
# Summary. The pilot in one table from the infos tables (no pair is read), the sizes in the archives, on
# disk and on Drive, the time of every step. All on Drive.
start = time.perf_counter()
meta = infos[list(pilotdata.META_COLUMNS)].copy()
meta["held_out"] = meta["grid_cell"].isin({pilotdata.pair_cell(p) for p in held_paths})
summary = pilotdata.pilot_summary(meta, skips)


def gb_of(folder, pattern):
    return sum(p.stat().st_size for p in Path(folder).glob(pattern) if p.is_file()) / stage0.GB


sizes = {"pairs": len(paths), "training_pairs": len(train_paths), "held_out_pairs": len(held_paths),
         "aerial_patches": {name: manifest["sets"][name]["count"] for name in aerial.SETS},
         "pairs_in_archives_gb": float(infos["bytes"].sum()) / stage0.GB,
         "pair_archives_on_drive_gb": gb_of(DRIVE_MT, "*.tar"),
         "aerial_sets_on_disk_gb": gb_of(SETS_DIR, "*.npz"),
         "aerial_archives_on_drive_gb": gb_of(DRIVE_AERIAL, "*.tar")}
print("\nsizes")
for key, value in sizes.items():
    print(f"  {key:<30} {value if isinstance(value, (int, dict)) else f'{value:,.2f}'}")
print("\ntime of every step of this run")
show(pd.DataFrame({"step": list(TIMES), "seconds": [round(v, 1) for v in TIMES.values()]}))
stage0.save_report(PILOT, pilot_summary=summary, pilot_meta=meta, sizes=sizes, times=TIMES)
print()
aerial.print_attribution()
print(f"\nfiles on Drive under {PILOT}")
for path in sorted(PILOT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(PILOT).as_posix():<40} {path.stat().st_size / stage0.MB:>10.1f} MB")
print(took(start, "summary"))


## 12. Upload to the private Hugging Face repo

Copies the archives, the lists and the statistics to the private repo. Needs HF_WRITE_TOKEN in the .env file. Files the repo already holds are skipped.

In [ ]:
# Upload. Everything the next notebooks read goes to the private Hugging Face repo too, with the write
# token from the .env file, hub.UPLOAD_WORKERS files at once (the reason is written there), the MB per
# second printed per file and in all. A file already there with the same size is skipped, a file that
# fails is named at the end and the others still go up. Without the token the cell says so and goes on,
# Drive holds everything.
start = time.perf_counter()
WRITE_TOKEN = os.environ.get("HF_WRITE_TOKEN")
if not WRITE_TOKEN:
    print("no HF_WRITE_TOKEN in the .env file, nothing is uploaded. Drive holds everything and"
          " notebooks 02 and 03 read it from there. Put a write token into the file and run this cell"
          f" again to fill {DATA_REPO}")
else:
    try:
        print(f"{hub.UPLOAD_WORKERS} files at once")
        rid = hub.upload_pilot(PILOT, token=WRITE_TOKEN)
        listing = hub.list_pilot(token=WRITE_TOKEN)
        print(f"\nfiles in {rid}")
        for name, size in sorted(listing.items()):
            print(f"  {name:<40} {size / stage0.MB:>10.1f} MB")
        print(f"total {sum(listing.values()) / stage0.GB:.2f} GB in {len(listing)} files")
    except Exception as err:
        print(f"UPLOAD FAILED, {type(err).__name__}, {hide(err)[:300]}. Drive holds everything. Run this"
              " cell again later, files already on the Hub with the same size are skipped")
del WRITE_TOKEN
print(took(start, "upload"))

## 13. Flush Drive

Waits until Drive holds everything, then unmounts it. This is the sign the run is over. To do anything on Drive after it, run the mount cell again.

In [ ]:
# Flush Drive. The run is over when this cell prints, Drive then holds everything that was written.
# Drive is unmounted by it. Run the mount cell again to do anything else on Drive in this runtime.
from google.colab import drive

drive.flush_and_unmount()
print("Drive holds everything and is unmounted. Run the mount cell again to do anything else on Drive")
